# 英国交通事故严重程度分类——完整流程


In [ ]:
# ============================================================================
# 设置：导入、路径与配置
# ============================================================================

import sys
import os
import json
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

import config
from src.data_loading import (
    load_accidents, load_vehicles, parse_datetime,
    aggregate_vehicles, merge_accidents_vehicles, load_context_tables
)
from src.data_cleaning import (
    identify_missing_patterns, drop_high_missing_columns,
    remove_leakage_columns, clip_outliers, asymmetric_undersample,
    generate_cleaning_report
)
from src.feature_engineering import (
    create_temporal_features, create_binary_features,
    create_interaction_features, frequency_encode_local_authority,
    get_feature_lists, build_preprocessing_pipeline
)
from src.clustering import (
    prepare_clustering_data, find_optimal_k, fit_kmeans,
    fit_agglomerative, compute_cluster_profiles,
    cluster_severity_crosstab, compute_pca_2d,
    plot_elbow, plot_silhouette, plot_pca_clusters,
    plot_cluster_profiles, plot_cluster_severity, plot_dendrogram
)
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from src.models import get_model_configs
from src.evaluation import (
    compute_metrics, plot_confusion_matrix, plot_roc_curves,
    plot_pr_curves, plot_feature_importance, plot_model_comparison,
    create_comparison_table, evaluate_temporal_holdout,
    analyze_cluster_errors
)

%matplotlib inline
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

# 相对项目根目录解析输出路径（不是笔记本目录）
FIGURES_DIR = os.path.join(PROJECT_ROOT, config.FIGURES_DIR)
RESULTS_DIR = os.path.join(PROJECT_ROOT, config.RESULTS_DIR)

print(f"Project root: {PROJECT_ROOT}")
print(f"Figures dir:  {FIGURES_DIR}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Config: FULL_SCALE_RUN={config.FULL_SCALE_RUN}, SEED={config.RANDOM_SEED}")

## 阶段1–3：数据加载、车辆聚合与清洗

In [ ]:
# ============================================================================
# 阶段1：加载原始CSV，将-1替换为NaN，解析日期/时间
# ============================================================================

start_time = time.time()

accidents_df, minus_one_counts = load_accidents(config.ACCIDENTS_FILE)
vehicles_df = load_vehicles(config.VEHICLES_FILE)
accidents_df = parse_datetime(accidents_df)
context_tables = load_context_tables(config.CONTEXT_DIR)

print(f"Accidents shape: {accidents_df.shape}")
print(f"Vehicles shape:  {vehicles_df.shape}")
print(f"Context tables:  {len(context_tables)} loaded")
print(f"\nTarget distribution (Accident_Severity):")
severity_dist = accidents_df['Accident_Severity'].value_counts().sort_index()
severity_labels = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
for code, count in severity_dist.items():
    pct = count / len(accidents_df) * 100
    print(f"  {code} ({severity_labels.get(code, code)}): {count:,} ({pct:.2f}%)")

print(f"\nColumns with -1 values (top 10):")
for col, cnt in sorted(minus_one_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {col}: {cnt:,} ({cnt/len(accidents_df)*100:.2f}%)")

In [ ]:
# ============================================================================
# 阶段3 -> 阶段2：先聚合车辆再清洗
# ============================================================================
vehicles_agg = aggregate_vehicles(vehicles_df)
merged_df = merge_accidents_vehicles(accidents_df, vehicles_agg)
print(f"After vehicle merge: {merged_df.shape}")
print(f"Vehicle aggregate features: {list(vehicles_agg.columns)}")

# 阶段2：清洗流程
# 步骤：删除高缺失列 -> 移除泄露列 -> 截断异常值
df_before = merged_df.copy()
missing_report = identify_missing_patterns(merged_df)
cleaned_df, dropped_cols = drop_high_missing_columns(merged_df)
print(f"\nDropped {len(dropped_cols)} high-missing columns: {dropped_cols}")

cleaned_df = remove_leakage_columns(cleaned_df)
cleaned_df = clip_outliers(cleaned_df)
cleaning_report = generate_cleaning_report(df_before, cleaned_df)

print(f"After cleaning: {cleaned_df.shape}")
print(f"Remaining NaN rates (top 5):")
nan_rates = (cleaned_df.isna().sum() / len(cleaned_df) * 100).sort_values(ascending=False)
for col, pct in nan_rates.head(5).items():
    if pct > 0:
        print(f"  {col}: {pct:.2f}%")

print(f"\nPhase 1-3 complete in {time.time() - start_time:.1f}s")

## 阶段4：特征工程


In [ ]:
# ============================================================================
# 阶段4：特征工程（与标签无关）
# ============================================================================

featured_df = create_temporal_features(cleaned_df)
featured_df = create_binary_features(featured_df, context_tables)
featured_df = create_interaction_features(featured_df)

print(f"Featured shape: {featured_df.shape}")
print(f"\nFeature list ({featured_df.shape[1]} columns):")

# 按类型分组展示特征
feature_lists = get_feature_lists()
for group_name, cols in feature_lists.items():
    present = [c for c in cols if c in featured_df.columns]
    print(f"  {group_name}: {len(present)} features — {present}")

# 对工程特征做快速合理性检查
eng_cols = ['Is_Dark', 'Bad_Weather', 'Wet_Road', 'Is_Rural', 'At_Junction',
            'Is_Weekend', 'Dark_And_Wet', 'Rural_High_Speed', 'Dark_Rural']
for col in eng_cols:
    if col in featured_df.columns:
        print(f"  {col}: mean={featured_df[col].mean():.3f}")

## 阶段5–7：训练/测试划分、非对称采样、频率编码



In [ ]:
# ============================================================================
# 阶段5：分层训练/测试划分（80/20）
# ============================================================================

target = 'Accident_Severity'
X = featured_df.drop(columns=[target])
y = featured_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config.TEST_SIZE,
    stratify=y,
    random_state=config.RANDOM_SEED
)

# 保留未采样的训练数据用于方案A消融
X_train_full = X_train.copy()
y_train_full = y_train.copy()

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nTrain class distribution:")
for code, pct in (y_train.value_counts(normalize=True).sort_index() * 100).items():
    print(f"  {severity_labels.get(code, code)}: {pct:.2f}%")

# ============================================================================
# 阶段6：非对称欠采样（仅训练集）
# ============================================================================

train_df = pd.concat([X_train, y_train], axis=1)
train_sampled = asymmetric_undersample(
    train_df,
    slight_ratio=config.SLIGHT_SAMPLE_RATIO,
    random_state=config.RANDOM_SEED
)
X_train_sampled = train_sampled.drop(columns=[target])
y_train_sampled = train_sampled[target]

print(f"\nBefore sampling: {len(X_train):,}")
print(f"After sampling:  {len(X_train_sampled):,}")
print(f"Sampled distribution:")
for code, count in y_train_sampled.value_counts().sort_index().items():
    print(f"  {severity_labels.get(code, code)}: {count:,}")

# ============================================================================
# 阶段7：频率编码（地方行政区）
# ============================================================================

X_train_encoded, X_test_encoded, la_freq_map = frequency_encode_local_authority(
    X_train_sampled, X_test
)

# 同时为未采样训练集做编码以用于消融
la_col = 'Local_Authority_(District)'
X_train_full_enc = X_train_full.copy()
if la_col in X_train_full_enc.columns:
    global_mean_freq = 1.0 / len(la_freq_map)
    X_train_full_enc['LA_Frequency'] = X_train_full_enc[la_col].map(
        la_freq_map).fillna(global_mean_freq)
    X_train_full_enc = X_train_full_enc.drop(columns=[la_col])

print(f"\nEncoded train shape: {X_train_encoded.shape}")
print(f"Encoded test shape:  {X_test_encoded.shape}")
print(f"LA_Frequency range: [{X_train_encoded['LA_Frequency'].min():.6f}, "
      f"{X_train_encoded['LA_Frequency'].max():.6f}]")

## 阶段8：聚类EDA（K=4）



In [ ]:
# ============================================================================
# 阶段8：聚类EDA
# ============================================================================
OPTIMAL_K = 4

# 步骤1：为K搜索准备子样本（加速）
scaled_sample, _, sample_idx_sub = prepare_clustering_data(
    X_train_encoded,
    feature_cols=config.CLUSTER_FEATURES,
    sample_size=config.CLUSTER_SAMPLE_SIZE,
    random_state=config.RANDOM_SEED
)

# 步骤2：用肘部法 + 轮廓系数寻找最优K
k_results = find_optimal_k(
    scaled_sample,
    k_range=config.CLUSTER_K_RANGE,
    random_state=config.RANDOM_SEED
)
plot_elbow(k_results, save_path=os.path.join(FIGURES_DIR, 'Fig04_elbow_plot.png'))
plot_silhouette(k_results, save_path=os.path.join(FIGURES_DIR, 'Fig05_silhouette_scores.png'))

# 步骤3：在完整训练集上用K=4重拟合
full_scaled, cluster_scaler, full_idx = prepare_clustering_data(
    X_train_encoded,
    feature_cols=config.CLUSTER_FEATURES,
    sample_size=None,
    random_state=config.RANDOM_SEED
)
kmeans_model, kmeans_labels = fit_kmeans(full_scaled, OPTIMAL_K, config.RANDOM_SEED)

X_full = X_train_encoded.iloc[full_idx]
y_full = y_train_sampled.iloc[full_idx]

# 步骤4：聚类画像与严重度交叉表
profiles = compute_cluster_profiles(X_full, kmeans_labels, config.CLUSTER_FEATURES)
ct_counts, ct_props = cluster_severity_crosstab(y_full, kmeans_labels)

# 步骤5：PCA可视化
VIZ_SAMPLE = 80000
rng = np.random.RandomState(config.RANDOM_SEED)
if len(full_scaled) > VIZ_SAMPLE:
    viz_idx = rng.choice(len(full_scaled), size=VIZ_SAMPLE, replace=False)
    viz_scaled, viz_labels = full_scaled[viz_idx], kmeans_labels[viz_idx]
else:
    viz_scaled, viz_labels = full_scaled, kmeans_labels

pca_2d, var_ratio, _ = compute_pca_2d(viz_scaled, config.RANDOM_SEED)
plot_pca_clusters(pca_2d, viz_labels, var_ratio,
                  save_path=os.path.join(FIGURES_DIR, 'Fig06_pca_clusters.png'))
plot_cluster_profiles(profiles,
                      save_path=os.path.join(FIGURES_DIR, 'Fig07_cluster_profiles_heatmap.png'))
plot_cluster_severity(ct_counts, ct_props,
                      save_path=os.path.join(FIGURES_DIR, 'Fig08_cluster_severity_crosstab.png'))

# 步骤6：层次聚类对比
AGG_SAMPLE_SIZE = 20000
agg_scaled, _, agg_idx = prepare_clustering_data(
    X_train_encoded, config.CLUSTER_FEATURES, AGG_SAMPLE_SIZE, config.RANDOM_SEED)

_, km_agg_labels = fit_kmeans(agg_scaled, OPTIMAL_K, config.RANDOM_SEED)
_, agg_labels = fit_agglomerative(agg_scaled, OPTIMAL_K)
ari = adjusted_rand_score(km_agg_labels, agg_labels)
print(f"K-Means vs Agglomerative ARI (K={OPTIMAL_K}): {ari:.4f}")

# 步骤7：树状图
plot_dendrogram(agg_scaled, max_samples=5000, random_state=config.RANDOM_SEED,
                save_path=os.path.join(FIGURES_DIR, 'Fig09_dendrogram.png'))

print(f"\nPhase 8 complete: K={OPTIMAL_K}, {len(kmeans_labels):,} samples clustered")

In [ ]:
# 内联显示聚类图
fig_names = [
    ('Fig04_elbow_plot.png', 'Elbow Plot (SSE vs K)'),
    ('Fig05_silhouette_scores.png', 'Silhouette Score vs K'),
    ('Fig06_pca_clusters.png', 'PCA 2D Scatter (K=4)'),
    ('Fig07_cluster_profiles_heatmap.png', 'Cluster Feature Profiles'),
    ('Fig08_cluster_severity_crosstab.png', 'Cluster x Severity Distribution'),
    ('Fig09_dendrogram.png', 'Agglomerative Dendrogram'),
]
for fname, title in fig_names:
    fpath = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(fpath):
        print(f"\n--- {title} ---")
        display(Image(filename=fpath, width=600))
    else:
        print(f"WARNING: {fname} not found at {fpath}")

## 阶段9–10：基线模型与Cluster_ID比较



In [ ]:
# ============================================================================
# 阶段9：不含Cluster_ID的基线模型
# ============================================================================

model_configs = get_model_configs(include_cluster=False)
baseline_results = {}

for name, cfg in model_configs.items():
    t0 = time.time()
    print(f"Training {name}...")
    pipeline = cfg['pipeline']
    pipeline.fit(X_train_encoded, y_train_sampled)
    y_pred = pipeline.predict(X_test_encoded)
    y_proba = pipeline.predict_proba(X_test_encoded) if hasattr(pipeline, 'predict_proba') else None
    metrics = compute_metrics(y_test, y_pred, y_proba,
                              class_names=['Fatal', 'Serious', 'Slight'])
    baseline_results[name] = {'metrics': metrics, 'pipeline': pipeline,
                               'y_pred': y_pred, 'y_proba': y_proba}
    elapsed = time.time() - t0
    print(f"  Macro F1: {metrics.get('macro_f1', 'N/A'):.4f}, "
          f"Fatal Recall: {metrics.get('Fatal_recall', 'N/A'):.4f}, "
          f"Balanced Acc: {metrics.get('balanced_accuracy', 'N/A'):.4f} "
          f"({elapsed:.1f}s)")

print("\n--- Baseline Results Summary ---")
summary_rows = []
for name, res in baseline_results.items():
    m = res['metrics']
    summary_rows.append({
        'Model': name,
        'Accuracy': f"{m['accuracy']:.4f}",
        'Balanced Acc': f"{m['balanced_accuracy']:.4f}",
        'Macro F1': f"{m['macro_f1']:.4f}",
        'Fatal Recall': f"{m['Fatal_recall']:.4f}",
        'Fatal Precision': f"{m['Fatal_precision']:.4f}",
    })
display(pd.DataFrame(summary_rows).set_index('Model'))

In [ ]:
# ============================================================================
# 阶段10：包含Cluster_ID的增强模型
# ============================================================================

model_configs_cluster = get_model_configs(include_cluster=True)
enhanced_results = {}

for name, cfg in model_configs_cluster.items():
    t0 = time.time()
    print(f"Training {name} + Cluster_ID...")
    pipeline = cfg['pipeline']
    pipeline.fit(X_train_encoded, y_train_sampled)
    y_pred = pipeline.predict(X_test_encoded)
    y_proba = pipeline.predict_proba(X_test_encoded) if hasattr(pipeline, 'predict_proba') else None
    metrics = compute_metrics(y_test, y_pred, y_proba,
                              class_names=['Fatal', 'Serious', 'Slight'])
    enhanced_results[name] = {'metrics': metrics, 'pipeline': pipeline}
    elapsed = time.time() - t0
    print(f"  Macro F1: {metrics.get('macro_f1', 'N/A'):.4f}, "
          f"Fatal Recall: {metrics.get('Fatal_recall', 'N/A'):.4f} "
          f"({elapsed:.1f}s)")

# Cluster_ID提升对比
print("\n--- Cluster_ID Uplift Comparison ---")
uplift_rows = []
for name in baseline_results:
    b = baseline_results[name]['metrics']
    e = enhanced_results[name]['metrics']
    uplift_rows.append({
        'Model': name,
        'Baseline Macro F1': f"{b['macro_f1']:.4f}",
        'Cluster Macro F1': f"{e['macro_f1']:.4f}",
        'Delta': f"{e['macro_f1'] - b['macro_f1']:+.4f}",
        'Baseline Fatal Recall': f"{b['Fatal_recall']:.4f}",
        'Cluster Fatal Recall': f"{e['Fatal_recall']:.4f}",
    })
display(pd.DataFrame(uplift_rows).set_index('Model'))

## 阶段11：超参数调优与不平衡消融



In [ ]:
# ============================================================================
# 阶段11a：从缓存加载调参结果
# ============================================================================

tuning_path = os.path.join(RESULTS_DIR, 'tuning_checkpoint.json')
if os.path.exists(tuning_path):
    with open(tuning_path, 'r') as f:
        tuning_checkpoint = json.load(f)
    
    print("=== Hyperparameter Tuning Results (Cached) ===")
    print(f"Imbalance scheme: {tuning_checkpoint.get('scheme', 'N/A')}\n")
    
    for model_name, result in tuning_checkpoint['completed'].items():
        print(f"{model_name}:")
        print(f"  Best CV Macro F1: {result['best_score']:.4f}")
        print(f"  Best params: {result['best_params']}")
        print(f"  Search time: {result['elapsed_min']:.1f} min")
        print()
else:
    print(f"WARNING: Tuning checkpoint not found at {tuning_path}")
    print("Run 'python run_tuning.py' from the project root to generate.")

# 加载调优提升表
improvement_path = os.path.join(RESULTS_DIR, 'Table_tuning_improvement.csv')
if os.path.exists(improvement_path):
    improvement_df = pd.read_csv(improvement_path)
    print("\n--- Tuning Improvement (Default vs Tuned CV Macro F1) ---")
    display(improvement_df)
else:
    print(f"WARNING: {improvement_path} not found.")

In [ ]:
# ============================================================================
# 阶段11b：从缓存加载消融结果
# ============================================================================

ablation_path = os.path.join(RESULTS_DIR, 'Table_ablation.csv')
if os.path.exists(ablation_path):
    ablation_df = pd.read_csv(ablation_path)
    print("=== Imbalance Ablation Results (Cached) ===")
    
    # 展示每个方案×模型的关键指标
    cols_display = ['scheme', 'model', 'accuracy', 'balanced_accuracy', 'macro_f1',
                    'Fatal_recall', 'Fatal_precision', 'Serious_recall', 'Slight_recall']
    cols_present = [c for c in cols_display if c in ablation_df.columns]
    display(ablation_df[cols_present].round(4))
else:
    print(f"WARNING: Ablation table not found at {ablation_path}")
    print("Run 'python run_ablation.py' from the project root to generate.")

# 显示消融图
ablation_figs = [
    ('Fig16_ablation_comparison.png', 'Ablation: Scheme Comparison by Model'),
    ('Fig17_ablation_tradeoff.png', 'Ablation: Fatal Recall vs Precision Trade-off'),
    ('Fig18_ablation_cv_stability.png', 'Ablation: CV Stability Across Schemes'),
]
for fname, title in ablation_figs:
    fpath = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(fpath):
        print(f"\n--- {title} ---")
        display(Image(filename=fpath, width=700))
    else:
        print(f"WARNING: {fname} not found.")

## 阶段12：最终评估



In [ ]:
# ============================================================================
# 阶段12a：测试集结果
# ============================================================================

test_results_path = os.path.join(RESULTS_DIR, 'Table_test_results.csv')
if os.path.exists(test_results_path):
    test_results_df = pd.read_csv(test_results_path)
    print("=== L2: Test Set Results (Best Tuned Models, Scheme C) ===")
    cols_display = ['Model', 'accuracy', 'balanced_accuracy', 'macro_f1',
                    'Fatal_recall', 'Fatal_precision', 'Serious_recall',
                    'Slight_recall', 'auc_roc_macro']
    cols_present = [c for c in cols_display if c in test_results_df.columns]
    display(test_results_df[cols_present].round(4))
else:
    print(f"WARNING: Test results not found at {test_results_path}")
    print("Run 'python run_phase12.py' from the project root to generate.")

In [ ]:
# ============================================================================
# 阶段12b：三层对比（交叉验证/测试/时间切分）
# ============================================================================

comparison_path = os.path.join(RESULTS_DIR, 'Table_cv_vs_test_vs_temporal.csv')
if os.path.exists(comparison_path):
    comparison_df = pd.read_csv(comparison_path)
    print("=== L1/L2/L3 Three-Layer Evaluation Comparison ===")
    display(comparison_df.round(4))
    
    print("\nKey observations:")
    for _, row in comparison_df.iterrows():
        model = row['model']
        cv_test_gap = row.get('CV_vs_test', 0)
        test_temp_gap = row.get('test_vs_temporal', 0)
        print(f"  {model}: CV->Test gap={cv_test_gap:+.4f}, "
              f"Test->Temporal gap={test_temp_gap:+.4f}")
else:
    print(f"WARNING: Comparison table not found at {comparison_path}")

In [ ]:
# ============================================================================
# 阶段12c：Cluster_ID提升（最终调优模型）
# ============================================================================

cluster_cmp_path = os.path.join(RESULTS_DIR, 'Table_cluster_id_comparison.csv')
if os.path.exists(cluster_cmp_path):
    cluster_cmp_df = pd.read_csv(cluster_cmp_path)
    print("=== Cluster_ID Uplift (Baseline vs Cluster-Enhanced) ===")
    display(cluster_cmp_df.round(4))
    
    print("\nConclusion: Cluster_ID provides near-zero uplift across all models.")
    print("The 4 accident archetypes are already captured by the raw features")
    print("(Is_Dark, Is_Rural, Speed_limit, etc.) that define them.")
else:
    print(f"WARNING: Cluster comparison not found at {cluster_cmp_path}")

In [ ]:
# ============================================================================
# 阶段12d：聚类×预测错误分析
# ============================================================================

cluster_error_path = os.path.join(RESULTS_DIR, 'Table_cluster_error_analysis.csv')
fatal_misclass_path = os.path.join(RESULTS_DIR, 'Table_fatal_misclass_pattern.csv')

if os.path.exists(cluster_error_path):
    cluster_error_df = pd.read_csv(cluster_error_path)
    print("=== Cluster x Prediction Error Analysis ===")
    display(cluster_error_df.round(4))
else:
    print(f"WARNING: Cluster error analysis not found at {cluster_error_path}")

if os.path.exists(fatal_misclass_path):
    fatal_misclass_df = pd.read_csv(fatal_misclass_path)
    print("\n=== Fatal Misclassification Patterns by Cluster ===")
    display(fatal_misclass_df)
    
    print("\nInterpretation:")
    print("  Cluster 3 (rural high-speed dark) has the highest Fatal correct rate (38.4%),")
    print("  likely because its feature profile (high speed + dark + rural) strongly")
    print("  correlates with severity. Cluster 0 (urban daytime) is hardest for Fatal")
    print("  prediction (1.1% correct) because Fatal accidents in benign conditions")
    print("  lack distinguishing feature signatures.")
else:
    print(f"WARNING: Fatal misclassification patterns not found at {fatal_misclass_path}")

## 阶段13：关键可视化



In [ ]:
# ============================================================================
# 探索性分析图（Fig01-03）
# ============================================================================
print("=== Exploratory Data Analysis ===")

eda_figs = [
    ('Fig01_target_distribution.png', 'Target Distribution (Accident_Severity)'),
    ('Fig02_correlation_heatmap.png', 'Feature Correlation Heatmap'),
    ('Fig03_feature_severity_dist.png', 'Key Feature x Severity Distributions'),
]
for fname, title in eda_figs:
    fpath = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(fpath):
        print(f"\n--- {title} ---")
        display(Image(filename=fpath, width=700))
    else:
        print(f"WARNING: {fname} not found at {fpath}")

In [ ]:
# ============================================================================
# 模型评估图（Fig10-15）
# ============================================================================
print("=== Model Evaluation ===")

model_figs = [
    ('Fig10_confusion_matrix_RandomForest.png', 'Confusion Matrix: Random Forest'),
    ('Fig10_confusion_matrix_HistGBT.png', 'Confusion Matrix: HistGBT'),
    ('Fig10_confusion_matrix_LogisticRegression.png', 'Confusion Matrix: Logistic Regression'),
    ('Fig11_model_comparison.png', 'Model Comparison (Multi-Metric)'),
    ('Fig12_roc_curves.png', 'ROC Curves Comparison'),
    ('Fig13_pr_curves.png', 'Precision-Recall Curves'),
    ('Fig14_feature_importance_RandomForest.png', 'Feature Importance: Random Forest (Top 15)'),
    ('Fig14_feature_importance_HistGBT.png', 'Feature Importance: HistGBT (Top 15)'),
    ('Fig15_lr_coefficients.png', 'Logistic Regression Coefficients'),
]
for fname, title in model_figs:
    fpath = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(fpath):
        print(f"\n--- {title} ---")
        display(Image(filename=fpath, width=700))
    else:
        print(f"WARNING: {fname} not found at {fpath}")

In [ ]:
# ============================================================================
# 调参与综合图（Fig19-21 + 聚类错误）
# ============================================================================
print("=== Tuning & Synthesis ===")

tuning_figs = [
    ('Fig19_tuning_sensitivity_RandomForest.png', 'Tuning Sensitivity: Random Forest'),
    ('Fig19_tuning_sensitivity_HistGBT.png', 'Tuning Sensitivity: HistGBT'),
    ('Fig19_tuning_sensitivity_LogisticRegression.png', 'Tuning Sensitivity: Logistic Regression'),
    ('Fig20_tuning_cv_boxplot.png', 'CV Score Distribution Boxplot'),
    ('Fig21_tuning_before_after.png', 'Tuning: Default vs Tuned Performance'),
]
for fname, title in tuning_figs:
    fpath = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(fpath):
        print(f"\n--- {title} ---")
        display(Image(filename=fpath, width=700))
    else:
        print(f"WARNING: {fname} not found at {fpath}")

In [ ]:
# ============================================================================
# 聚类错误分布图
# ============================================================================

cluster_error_fig = os.path.join(FIGURES_DIR, 'Fig_cluster_error_distribution.png')
if os.path.exists(cluster_error_fig):
    print("--- Cluster x Prediction Error Distribution ---")
    display(Image(filename=cluster_error_fig, width=700))
else:
    print(f"WARNING: Cluster error figure not found at {cluster_error_fig}")